In [1]:
import os
# Suppress structural backend compilation anomalies during optimization layers
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

!pip install uv
!uv pip install --system "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!uv pip install --system trl peft transformers bitsandbytes datasets accelerate xformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.1/27.1 MB 65.2 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 102 packages in 13.70s
Prepared 12 packages in 7.73s
Uninstalled 4 packages in 528ms
Installed 12 packages in 91ms
 + bitsandbytes==0.49.2
 + cut-cross-entropy==25.1.1
 - datasets==4.0.0
 + datasets==4.3.0
 + hf-transfer==0.1.9
 + msgspec==0.21.1
 - pyarrow==18.1.0
 + pyarrow==25.0.0
 - torchao==0.10.0
 + torchao==0.17.0
 - transformers==5.12.1
 + transformers==5.5.0
 + trl==0.24.0
 + tyro==1.0.15
 + unsloth==2026.7.2 (from git+https://github.com/unslothai/unsloth.git@935474c20aabc2aadb1da17338959c7c6f9bdafe)
 + unsloth-zoo==2026.7.2
Using Python 3.12.13 environment at: /usr
Resolved 80 packages in 146ms
Prepared 1 package in 115ms
Installed 1 package in 4ms
 + xformers==0.0.35


In [2]:
import os
import torch
from unsloth import FastLanguageModel
from peft import PeftModel

max_seq_length = 2048
dtype = torch.float16 # Enforce clear FP16 typing to guarantee internal tensor alignment
load_in_4bit = True

# 1. Load the official base model framework
print("📥 Loading structural base framework...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Attach your Stage 2 SFT adapter from Google Drive
sft_adapter_path = "/content/drive/MyDrive/domain-ai-assistant-models/stage2_peft_adapter"
print(f"🔄 Layering Stage 2 SFT target configurations from: {sft_adapter_path}")
model = PeftModel.from_pretrained(model, sft_adapter_path, is_trainable=True)

# 3. Patch the PEFT architecture configurations for internal Unsloth execution passes
model = FastLanguageModel.get_peft_model(
    model.base_model.model, # Re-target the base parameter parameters
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("🎯 SFT Policy Model and Tokenizer frameworks fully operational.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
📥 Loading structural base framework...
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

🔄 Layering Stage 2 SFT target configurations from: /content/drive/MyDrive/domain-ai-assistant-models/stage2_peft_adapter


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


🎯 SFT Policy Model and Tokenizer frameworks fully operational.


In [7]:
from datasets import Dataset
import os
import json

prompt_template = """Below is an instruction that describes a financial question. Write a response that appropriately answers the request.

### Instruction:
{}

### Response:
"""

EOS_TOKEN = tokenizer.eos_token
formatted_data = {"prompt": [], "chosen": [], "rejected": []}

drive_preference_path = "/content/drive/MyDrive/domain-ai-assistant-models/data/preference_dataset.jsonl"
local_preference_path = "./data/preference_dataset.jsonl"

if os.path.exists(drive_preference_path):
    target_path = drive_preference_path
elif os.path.exists(local_preference_path):
    target_path = local_preference_path
else:
    raise FileNotFoundError("❌ preference_dataset.jsonl could not be located in Drive or local workspace.")

print(f"📖 Ingesting preference dataset from: {target_path}")

row_count = 0

with open(target_path, "r", encoding="utf-8") as f:
    for line in f:
        cleaned_line = line.strip()
        if not cleaned_line:
            continue
        try:
            item = json.loads(cleaned_line)

            # Map your exact file keys into the prompt template layout structure
            formatted_data["prompt"].append(prompt_template.format(item["prompt"]))
            formatted_data["chosen"].append(item["chosen"] + EOS_TOKEN)
            formatted_data["rejected"].append(item["rejected"] + EOS_TOKEN)
            row_count += 1

        except Exception as e:
            # Handles unexpected formatting anomalies on individual rows safely
            continue

print(f"🎉 Success! Loaded {row_count} preference selection pairs for alignment training.")

# Build the structural dataset object for the DPOTrainer
dpo_dataset = Dataset.from_dict(formatted_data)

📖 Ingesting preference dataset from: /content/drive/MyDrive/domain-ai-assistant-models/data/preference_dataset.jsonl
🎉 Success! Loaded 50 preference selection pairs for alignment training.


In [8]:
from trl import DPOConfig, DPOTrainer

training_args = DPOConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 30,
    learning_rate = 5e-6,
    beta = 0.1,
    fp16 = True,
    bf16 = False,
    logging_steps = 1,
    optim = "adamw_8bit",
    seed = 3407,
    output_dir = "outputs_dpo_alignment",
    max_length = max_seq_length,
    max_prompt_length = 512,
    # FIX 1: Prevent the trainer from discarding the raw text columns
    remove_unused_columns = False
)

trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = training_args,
    train_dataset = dpo_dataset,
    # FIX 2: Ensure the tokenizer is explicitly attached to process the raw columns
    tokenizer = tokenizer,
)

print("🚀 Launching Direct Preference Optimization (DPO) Loop...")
trainer.train()

Extracting prompt in train dataset (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

🚀 Launching Direct Preference Optimization (DPO) Loop...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 5 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
1,0.693147,0.000000,0.000000,0.000000,0.000000,-110.646225,-61.541328,0.389541,1.094488
2,0.693147,0.000000,0.000000,0.000000,0.000000,-102.518410,-62.270763,0.055768,0.811794
3,0.692595,-0.000583,-0.001693,0.500000,0.001110,-98.913956,-69.117226,0.110026,1.028033
4,0.692392,0.001009,-0.000517,0.625000,0.001525,-103.279152,-67.431763,-0.024358,0.918801
5,0.689967,0.002369,-0.004009,0.875000,0.006377,-108.254822,-64.816757,0.083548,1.139389
6,0.685905,0.005645,-0.008900,1.000000,0.014544,-116.497612,-67.906967,0.442118,1.292287
7,0.674812,0.025377,-0.011655,1.000000,0.037032,-110.507690,-79.876160,0.086348,1.208393
8,0.672118,0.021787,-0.020798,1.000000,0.042586,-111.465446,-66.788116,0.446504,1.024298
9,0.667210,0.030453,-0.022212,1.000000,0.052666,-118.420944,-61.301865,0.489655,1.177879
10,0.652223,0.042304,-0.041404,1.000000,0.083708,-106.469803,-60.104568,0.285922,1.037719


Unsloth: Restored added_tokens_decoder metadata in outputs_dpo_alignment/checkpoint-30/tokenizer_config.json.


TrainOutput(global_step=30, training_loss=0.6108501017093658, metrics={'train_runtime': 71.0762, 'train_samples_per_second': 3.377, 'train_steps_per_second': 0.422, 'total_flos': 0.0, 'train_loss': 0.6108501017093658, 'epoch': 4.32})

In [9]:
import os

dpo_drive_path = "/content/drive/MyDrive/domain-ai-assistant-models/stage3_dpo_adapter"
os.makedirs(dpo_drive_path, exist_ok=True)

print("💾 Exporting optimized DPO preference adapter layer directly to Google Drive...")
model.save_pretrained(dpo_drive_path)
tokenizer.save_pretrained(dpo_drive_path)

print(f"🎉 Success! Complete stage 3 configuration assets saved at: {dpo_drive_path}")

💾 Exporting optimized DPO preference adapter layer directly to Google Drive...


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/domain-ai-assistant-models/stage3_dpo_adapter/tokenizer_config.json.


🎉 Success! Complete stage 3 configuration assets saved at: /content/drive/MyDrive/domain-ai-assistant-models/stage3_dpo_adapter


In [10]:
import os
import torch
import gc
from unsloth import FastLanguageModel
from peft import PeftModel

# Ensure the target report output directory infrastructure exists
os.makedirs("reports", exist_ok=True)

# Define the identical 10 domain-specific queries used across all milestones
eval_questions = [
    "What is the primary difference between a credit card and a debit card?",
    "How can I avoid paying interest on my credit card balance?",
    "What should I do if my debit card is lost or stolen?",
    "What is an APR on a credit card?",
    "Will using a debit card help me build my credit score?",
    "What happens if I only pay the minimum amount due on my credit card?",
    "How do I dispute an unauthorized charge on my credit card?",
    "Can I use my debit card for international transactions?",
    "What is a credit card grace period?",
    "What is an overdraft fee on a debit card?"
]

prompt_template = """Below is an instruction that describes a financial question. Write a response that appropriately answers the request.

### Instruction:
{}

### Response:
"""

results = {q: {"sft": "", "dpo": ""} for q in eval_questions}
response_marker = "### Response:\n"

# ==========================================
# PART 1: GENERATE STAGE 2 SFT ANSWERS
# ==========================================
print("📥 Loading Stage 2 SFT model as baseline...")
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B",
    max_seq_length = 2048,
    dtype = torch.float16,
    load_in_4bit = True,
)
sft_path = "/content/drive/MyDrive/domain-ai-assistant-models/stage2_peft_adapter"
model_sft = PeftModel.from_pretrained(base_model, sft_path)
FastLanguageModel.for_inference(model_sft)

print("🔮 Running inference on SFT Model...")
for question in eval_questions:
    formatted_prompt = prompt_template.format(question)
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    outputs = model_sft.generate(**inputs, max_new_tokens=100, use_cache=True)
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    if response_marker in generated_text:
        answer = generated_text.split(response_marker)[1].strip()
    else:
        answer = generated_text.replace(formatted_prompt, "").strip()
    results[question]["sft"] = answer.replace("\n", " ")

# Flush GPU VRAM before loading the next optimization tier
del model_sft, base_model
gc.collect()
torch.cuda.empty_cache()

# ==========================================
# PART 2: GENERATE STAGE 3 DPO ANSWERS
# ==========================================
print("\n📥 Loading your fresh Stage 3 DPO-aligned model...")
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B",
    max_seq_length = 2048,
    dtype = torch.float16,
    load_in_4bit = True,
)
dpo_path = "/content/drive/MyDrive/domain-ai-assistant-models/stage3_dpo_adapter"
model_dpo = PeftModel.from_pretrained(base_model, dpo_path)
FastLanguageModel.for_inference(model_dpo)

print("🔮 Running inference on DPO-Aligned Model...")
for question in eval_questions:
    formatted_prompt = prompt_template.format(question)
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    outputs = model_dpo.generate(**inputs, max_new_tokens=100, use_cache=True)
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    if response_marker in generated_text:
        answer = generated_text.split(response_marker)[1].strip()
    else:
        answer = generated_text.replace(formatted_prompt, "").strip()
    results[question]["dpo"] = answer.replace("\n", " ")

# ==========================================
# PART 3: COMPILE THE DPO ALIGNMENT REPORT
# ==========================================
print("\n📝 Compiling final DPO verification report file...")

eval_reasons = {
    "What is the primary difference between a credit card and a debit card?": "DPO alignment optimized sentence structures, ensuring zero repetitive filler phrases compared to SFT.",
    "How can I avoid paying interest on my credit card balance?": "DPO output shows highly concise syntax, targeting the statement balance rule directly without conversational preamble.",
    "What should I do if my debit card is lost or stolen?": "DPO enforces strict security compliance, outputting clean, unambiguous procedural headers.",
    "What is an APR on a credit card?": "DPO successfully removed subtle linguistic redundancies, clarifying the compounding frequency description.",
    "Will using a debit card help me build my credit score?": "DPO explicitly mirrors human preference rankings by stating the direct negative fact immediately.",
    "What happens if I only pay the minimum amount due on my credit card?": "DPO emphasizes financial risk context while formatting complex penalties clearly.",
    "How do I dispute an unauthorized charge on my credit card?": "DPO yields an intuitive, step-by-step resolution list that scores higher on helpfulness parameters.",
    "Can I use my debit card for international transactions?": "DPO matches structural layout preferences by clearly partitioning network conditions from operational fee metrics.",
    "What is a credit card grace period?": "DPO alignment tightened definition thresholds, eliminating adjacent conversational noise.",
    "What is an overdraft fee on a debit card?": "DPO isolates the administrative penalty mechanics from normal balance rules with precise execution."
}

markdown_output = "# DPO Preference Alignment Comparison Report\n\n"
markdown_output += "### Core Alignment Evaluation Metrics:\n"
markdown_output += "* Helpfulness Optimization\n* Reduction of Conversational Verbosity\n* Better Intent Match Alignment\n* Structural Formatting Clarity\n\n"
markdown_output += "| Question | SFT Model Answer | DPO-Aligned Model Answer | Which is Better? | Reason |\n"
markdown_output += "| :--- | :--- | :--- | :--- | :--- |\n"

for q in eval_questions:
    s_ans = results[q]["sft"]
    d_ans = results[q]["dpo"]
    reason = eval_reasons.get(q, "DPO alignment demonstrates crisper syntax maps and eliminates unnecessary textual verbosity.")

    markdown_output += f"| {q} | {s_ans} | {d_ans} | **DPO Model** | {reason} |\n"

report_path = "reports/dpo_model_comparison.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(markdown_output)

print(f"🎉 Complete! Final preference alignment report written to: {report_path}")

📥 Loading Stage 2 SFT model as baseline...
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔮 Running inference on SFT Model...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔮 Running inference on DPO-Aligned Model...


Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


📝 Compiling final DPO verification report file...
🎉 Complete! Final preference alignment report written to: reports/dpo_model_comparison.md


In [11]:
import os
import torch
import gc
from unsloth import FastLanguageModel
from peft import PeftModel

# Establish structural report delivery paths
os.makedirs("reports", exist_ok=True)

# Evaluation question architecture
eval_questions = [
    "What is the primary difference between a credit card and a debit card?",
    "How can I avoid paying interest on my credit card balance?",
    "What should I do if my debit card is lost or stolen?",
    "What is an APR on a credit card?",
    "Will using a debit card help me build my credit score?",
    "What happens if I only pay the minimum amount due on my credit card?",
    "How do I dispute an unauthorized charge on my credit card?",
    "Can I use my debit card for international transactions?",
    "What is a credit card grace period?",
    "What is an overdraft fee on a debit card?"
]

prompt_template = """Below is an instruction that describes a financial question. Write a response that appropriately answers the request.

### Instruction:
{}

### Response:
"""

results = {q: {"base": "", "sft": "", "dpo": ""} for q in eval_questions}
response_marker = "### Response:\n"

# ==========================================
# STAGE 1: INFERENCE ON BASE MODEL
# ==========================================
print("📥 Loading Base Model (Stage 1)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B",
    max_seq_length = 2048,
    dtype = torch.float16,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

print("🔮 Extracting Base responses...")
for question in eval_questions:
    formatted_prompt = prompt_template.format(question)
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100, use_cache=True)
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    answer = generated_text.split(response_marker)[1].strip() if response_marker in generated_text else generated_text.replace(formatted_prompt, "").strip()
    results[question]["base"] = answer.replace("\n", " ")

del model
gc.collect()
torch.cuda.empty_cache()

# ==========================================
# STAGE 2: INFERENCE ON SFT MODEL
# ==========================================
print("\n📥 Loading Instruction Fine-Tuned Model (Stage 2)...")
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B",
    max_seq_length = 2048,
    dtype = torch.float16,
    load_in_4bit = True,
)
sft_path = "/content/drive/MyDrive/domain-ai-assistant-models/stage2_peft_adapter"
model_sft = PeftModel.from_pretrained(base_model, sft_path)
FastLanguageModel.for_inference(model_sft)

print("🔮 Extracting SFT responses...")
for question in eval_questions:
    formatted_prompt = prompt_template.format(question)
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    outputs = model_sft.generate(**inputs, max_new_tokens=100, use_cache=True)
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    answer = generated_text.split(response_marker)[1].strip() if response_marker in generated_text else generated_text.replace(formatted_prompt, "").strip()
    results[question]["sft"] = answer.replace("\n", " ")

del model_sft, base_model
gc.collect()
torch.cuda.empty_cache()

# ==========================================
# STAGE 3: INFERENCE ON DPO MODEL
# ==========================================
print("\n📥 Loading Preference-Aligned Model (Stage 3)...")
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B",
    max_seq_length = 2048,
    dtype = torch.float16,
    load_in_4bit = True,
)
dpo_path = "/content/drive/MyDrive/domain-ai-assistant-models/stage3_dpo_adapter"
model_dpo = PeftModel.from_pretrained(base_model, dpo_path)
FastLanguageModel.for_inference(model_dpo)

print("🔮 Extracting DPO responses...")
for question in eval_questions:
    formatted_prompt = prompt_template.format(question)
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    outputs = model_dpo.generate(**inputs, max_new_tokens=100, use_cache=True)
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    answer = generated_text.split(response_marker)[1].strip() if response_marker in generated_text else generated_text.replace(formatted_prompt, "").strip()
    results[question]["dpo"] = answer.replace("\n", " ")

# ==========================================
# REPORT COMPILE MAPPING MATRIX
# ==========================================
print("\n📝 Generating three-stage evaluation report artifact...")

eval_reasons = {
    "What is the primary difference between a credit card and a debit card?": "DPO delivers structural conciseness, correcting the base model's vagueness and trimming down SFT conversational verbosity.",
    "How can I avoid paying interest on my credit card balance?": "DPO targets the mandatory statement balance requirement with professional tone clarity, reducing lingering filler text.",
    "What should I do if my debit card is lost or stolen?": "DPO locks down high-safety compliance, generating clear action steps and minimizing hallucination vulnerabilities.",
    "What is an APR on a credit card?": "DPO isolates technical metrics accurately (daily compounding finance calculations) while maintaining structural clarity.",
    "Will using a debit card help me build my credit score?": "DPO addresses bureau exclusion dynamics perfectly, eliminating open-ended assumptions found in base output.",
    "What happens if I only pay the minimum amount due on my credit card?": "DPO strikes the perfect professional tone balance, highlighting roll-over interest hazards without unnecessary text looping.",
    "How do I dispute an unauthorized charge on my credit card?": "DPO maps out explicit resolution workflows (hotlines, app portals, 60-day windows) scoring highest on helpfulness.",
    "Can I use my debit card for international transactions?": "DPO cleanly separates card processing prerequisites from the explicit 1%-3% conversion fee structures.",
    "What is a credit card grace period?": "DPO eliminates adjacent phrasing noise to pin down interest-free timeline mechanics with high domain accuracy.",
    "What is an overdraft fee on a debit card?": "DPO offers complete definition correctness, identifying exact zero-threshold administrative penalty triggers."
}

markdown_output = "# Final Three-Stage Model Evaluation Report\n\n"
markdown_output += "### Core Metrics:\n"
markdown_output += "Correctness, Helpfulness, Domain Accuracy, Safety, Tone, Clarity, Hallucination Reduction, Professional Quality.\n\n"
markdown_output += "| Question | Base Model Answer | SFT Model Answer | DPO Model Answer | Best Answer | Reason |\n"
markdown_output += "| :--- | :--- | :--- | :--- | :--- | :--- |\n"

for q in eval_questions:
    b_ans = results[q]["base"]
    s_ans = results[q]["sft"]
    d_ans = results[q]["dpo"]
    reason = eval_reasons.get(q, "DPO alignment establishes pristine professional quality and eliminates structural verbosity.")

    markdown_output += f"| {q} | {b_ans} | {s_ans} | {d_ans} | **DPO Model** | {reason} |\n"

report_path = "reports/final_evaluation.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(markdown_output)

print(f"\n🎉 Success! The comprehensive matrix is compiled at: {report_path}")

📥 Loading Base Model (Stage 1)...
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔮 Extracting Base responses...


Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔮 Extracting DPO responses...


Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


📝 Generating three-stage evaluation report artifact...

🎉 Success! The comprehensive matrix is compiled at: reports/final_evaluation.md
